In [3]:
from transformers import pipeline
import re
import json

# Load model
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=300,
    max_length=None,
    temperature=0.7,
    do_sample=True,
    return_full_text=False
)

MAX_TURNS = 5

# ---------- FORCE JSON OUTPUT ----------

INTRO_PROMPT = """You are an RPG story engine. Respond ONLY with valid JSON, nothing else.

Theme: {theme}
Character: {character}

JSON format (respond with ONLY this, no extra text):
{{
  "story": "3-4 sentence scene description with progression",
  "options": ["action one", "action two", "action three"]
}}"""

SCENE_PROMPT = """You are an RPG story engine.

Rules:
- Continue the story meaningfully (no repetition)
- Progress the plot forward (new events must happen)
- Introduce new characters, enemies, or discoveries
- Do NOT repeat previous scenes or phrases
- Increase tension each turn

Theme: {theme}
Character: {character}
Previous: {context}
Player did: {action}
{ending_note}

Respond ONLY with valid JSON:

{{
  "story": "3-4 sentence detailed continuation with NEW events",
  "options": [
  "action 1 based on the story",
  "action 2 based on the story",
  "action 3 based on the story"
 ]
}}"""

FALLBACK_OPTIONS = [
    "Move forward cautiously",
    "Look around carefully",
    "Wait and observe"
]


# ---------- CALL MODEL ----------
def call_model(prompt):
    result = generator(prompt)
    return result[0]['generated_text'].strip()


# ---------- EXTRACT JSON ----------
def extract_json(text):
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass

    story_match = re.search(r'"story"\s*:\s*"([^"]+)"', text)
    options_matches = re.findall(r'"([^"]{10,80})"', text)

    story = story_match.group(1) if story_match else None
    options = [o for o in options_matches if o != story][:3]

    if story:
        return {"story": story, "options": options if len(options) >= 3 else FALLBACK_OPTIONS}

    return None


# ---------- ENSURE OPTIONS ----------
def ensure_options(options):
    if not options or len(options) < 3:
        return FALLBACK_OPTIONS[:]
    return options[:3]


# ---------- GENERATE INTRO ----------
def generate_intro(theme, character):
    prompt = INTRO_PROMPT.format(theme=theme, character=character)
    for attempt in range(3):
        raw = call_model(prompt)
        data = extract_json(raw)
        if data and data.get("story") and len(data.get("options", [])) >= 3:
            return data["story"], data["options"][:3]
    return (
        f"A {character} stands at the edge of a {theme} world, destiny unknown.",
        FALLBACK_OPTIONS[:]
    )


# ---------- GENERATE SCENE ----------
def generate_scene(story, action, theme, character, is_last=False):
    ending_note = "This is the FINAL scene. End the story." if is_last else ""

    # 🔥 improved memory (recent context instead of start)
    context = story[-1200:]

    prompt = SCENE_PROMPT.format(
        theme=theme,
        character=character,
        context=context,
        action=action,
        ending_note=ending_note
    )

    for attempt in range(3):
        raw = call_model(prompt)
        data = extract_json(raw)

        if data and data.get("story") and len(data.get("options", [])) >= 3:
            new_story = data["story"]

            # 🔥 simple anti-repetition fix
            if story[-100:] in new_story:
                new_story = new_story.replace(story[-100:], "")

            return new_story, data["options"][:3]

    return (
        f"{story}\n\nYou chose: {action}. The journey continues...",
        FALLBACK_OPTIONS[:]
    )


# ---------- DISPLAY ----------
def show_story(story, turn, max_turns):
    print(f"\n{'='*55}")
    print(f"  Chapter {turn + 1} of {max_turns}")
    print(f"{'='*55}")
    print(f"\n{story}\n")


def show_options(options):
    print("  What do you do?\n")
    for i, opt in enumerate(options, 1):
        print(f"  [{i}] {opt}")
    print()


def get_choice(options):
    while True:
        try:
            choice = int(input("  Your choice (1/2/3): "))
            if 1 <= choice <= len(options):
                return options[choice - 1]
            print("  Please enter 1, 2, or 3.")
        except (ValueError, KeyboardInterrupt):
            print("  Invalid input, try again.")


# ---------- GAME LOOP ----------
while True:
    print("\n" + "=" * 55)
    print("           ⚔  AI STORY GAME  ⚔")
    print("=" * 55)

    name = input("\n  Character name: ").strip() or "Hero"
    theme = input("  Theme (fantasy / horror / sci-fi): ").strip() or "fantasy"
    role = input("  Role (warrior / hacker / detective): ").strip() or "warrior"
    character = f"{role} named {name}"

    print(f"\n  Loading your adventure as {character}...\n")

    story, options = generate_intro(theme, character)
    options = ensure_options(options)

    show_story(story, 0, MAX_TURNS)
    show_options(options)

    for turn in range(MAX_TURNS):
        chosen = get_choice(options)
        print(f"\n  ► You chose: {chosen}\n")
        print("  The story continues...\n")

        is_last = (turn == MAX_TURNS - 1)
        story, options = generate_scene(story, chosen, theme, character, is_last)
        options = ensure_options(options)

        show_story(story, turn + 1, MAX_TURNS)

        if is_last:
            print("  🎬 THE END\n")
            break

        show_options(options)

    print("=" * 55)
    again = input("  Play again? (yes / exit): ").strip().lower()
    if again != "yes":
        print("\n  Thanks for playing! ⚔\n")
        break

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


           ⚔  AI STORY GAME  ⚔

  Character name: Arjun
  Theme (fantasy / horror / sci-fi): fantasy
  Role (warrior / hacker / detective): warrior

  Loading your adventure as warrior named Arjun...


  Chapter 1 of 5

Arjun and his friends gather to plan a raid on a nearby city.

  What do you do?

  [1] join the raid
  [2] discuss the plan
  [3] prepare for the raid

  Your choice (1/2/3): 1

  ► You chose: join the raid

  The story continues...


  Chapter 2 of 5

Arjun leads his team into the city, preparing for the raid.

  What do you do?

  [1] : []
}
{
  
  [2] attack the passageway
  [3] find treasure in the city

  Your choice (1/2/3): 3

  ► You chose: find treasure in the city

  The story continues...


  Chapter 3 of 5

Arjun and his team face off against a group of bandits.

  What do you do?

  [1] Continue the fight to rescue the leader of the bandits
  [2] Find a way to communicate with the bandits without being seen
  [3] Tackle the bandits using their weapons

  